In [2]:
from petsc4py import PETSc;
from slepc4py import SLEPc;
import sys
import numpy as np;

In [2]:
%run test.py


******************************
*** SLEPc Solution Results ***
******************************

Number of iterations of the method: 4
Solution method: krylovschur
Number of requested eigenvalues: 1
Stopping condition: tol=1e-08, maxit=100
Number of converged eigenpairs 1

        k          ||Ax-kx||/||kx|| 
----------------- ------------------
     3.989739       4.72989e-09



In [4]:
import numpy as np
from scipy.integrate import solve_bvp
import matplotlib.pyplot as plt
kappa = -1
def oneDiskODE(z, y):

        # Y0 = H, Y1 = F', Y2 = F, Y3 = G', Y4 = G
        dydz = np.zeros((5, len(z)))
        dydz = np.array([-
                        2.0*
                        y[2], kappa * 
                        (y[2] *
                        y[2] +
                            y[0] *
                            y[1] -
                            (y[4] *
                            y[4]- 
                            1.0)) -
                        (2.0 -
                            kappa - 
                            kappa**2) *
                        (y[4] -
                            1.0), y[1], kappa *
                        (2.0 * 
                            y[2] *
                            y[4] +
                            y[0] *
                            y[3]) +
                        (2.0 -
                            kappa -
                            kappa**2) *
                        y[2], y[3]])
        return dydz 

def oneDiskBC(ya, yb):
        resa = np.array([ya[0],
                        ya[2],
                        ya[4]])
        
        resb = np.array([yb[2],
                        yb[4] - 1.0])
        
        return np.concatenate((resa, resb))


z = np.linspace(0, 30, 20000)
y = np.zeros((5, len(z)))
y_guess = np.zeros((5, z.size))
if kappa == 1:
        y_guess[0] = 1.2
        y_guess[1] = 0
        y_guess[2] = 0
        y_guess[3] = 0
        y_guess[4] = 1
elif kappa == -1:
        y_guess[0] = 1.2
        y_guess[1] = 0
        y_guess[2] = 0
        y_guess[3] = 0
        y_guess[4] = 1
else:
        y_guess[0] = 1.2
        y_guess[1] = 0
        y_guess[2] = 0
        y_guess[3] = 0
        y_guess[4] = 1




solution = solve_bvp(oneDiskODE, oneDiskBC, z, y_guess,tol=1e-10,max_nodes=5000000)

x_plot = np.linspace(0, 30, 20000)


y1_plot = solution.sol(x_plot)[0]
y2_plot = solution.sol(x_plot)[2]
y3_plot = solution.sol(x_plot)[4]
y4_plot = solution.sol(x_plot)[1]
y5_plot = solution.sol(x_plot)[3]

In [6]:
opts = PETSc.Options()
n = opts.getInt('n', 30)
A = PETSc.Mat().create()
A.setSizes([n, n])
A.setFromOptions()
A.setUp()

rstart, rend = A.getOwnershipRange()

# first row
if rstart == 0:
    A[0, :2] = [2, -1]
    rstart += 1
# last row
if rend == n:
    A[n-1, -2:] = [-1, 2]
    rend -= 1
# other rows
for i in range(rstart, rend):
    A[i, i-1:i+2] = [-1, 2, -1]

A.assemble()
E = SLEPc.EPS(); E.create()
E.setOperators(A)
E.setProblemType(SLEPc.EPS.ProblemType.HEP)
E.setFromOptions()
E.solve()
Print = PETSc.Sys.Print

Print()
Print("******************************")
Print("*** SLEPc Solution Results ***")
Print("******************************")
Print()

its = E.getIterationNumber()
Print("Number of iterations of the method: %d" % its)

eps_type = E.getType()
Print("Solution method: %s" % eps_type)

nev, ncv, mpd = E.getDimensions()
Print("Number of requested eigenvalues: %d" % nev)

tol, maxit = E.getTolerances()
Print("Stopping condition: tol=%.4g, maxit=%d" % (tol, maxit))
nconv = E.getConverged()
Print("Number of converged eigenpairs %d" % nconv)
if nconv > 0:
    # Create the results vectors
    vr, wr = A.getVecs()
    vi, wi = A.getVecs()
    #
    Print()
    Print("        k          ||Ax-kx||/||kx|| ")
    Print("----------------- ------------------")
    for i in range(nconv):
        k = E.getEigenpair(i, vr, vi)
        error = E.computeError(i)
        if k.imag != 0.0:
            Print(" %9f%+9f j %12g" % (k.real, k.imag, error))
        else:
            Print(" %12f      %12g" % (k.real, error))
    Print()



******************************
*** SLEPc Solution Results ***
******************************

Number of iterations of the method: 4
Solution method: krylovschur
Number of requested eigenvalues: 1
Stopping condition: tol=1e-08, maxit=100
Number of converged eigenpairs 1

        k          ||Ax-kx||/||kx|| 
----------------- ------------------
     3.989739       4.72989e-09

